# PageInex -- Vectorless RAG

In [3]:
import os, json, time

from dotenv import load_dotenv
load_dotenv()

os.environ['PAGEINDEX_API_KEY'] = os.getenv('PAGEINDEX_API_KEY')
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')


In [4]:
from pageindex import PageIndexClient
from langchain.chat_models import init_chat_model

pi_key = os.environ['PAGEINDEX_API_KEY']
pi_client = PageIndexClient(api_key = pi_key)
llm = init_chat_model(model='groq:qwen/qwen32b-3b')


In [5]:
# Uploading pdf to PageIndex Cloud

PDF_PATH = '../RAGS/data/pdf/Yash Prasad 229309105 Major Project Report V2.pdf'
result = pi_client.submit_document(PDF_PATH)
doc_id = result['doc_id']

print(f'Uploaded {PDF_PATH}\nDocument ID: {doc_id}')

Uploaded ../RAGS/data/pdf/Yash Prasad 229309105 Major Project Report V2.pdf
Document ID: pi-cmrsqnx5u01c801qut6d0bgic


In [6]:
# tree index
while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get('status')

    if status == 'completed':
        print('Tree index is ready')
        break
    elif status == 'failed':
        print('\n Processing Failed. Check pdf format.')
        break
    time.sleep(5)

Tree index is ready


Tree Structure <br>
Each node has:
- node_id: unique id used during retrieval
- title: section heading
- page_inde: page number in original PDF
- text: section summary (when node_summary = True)
- nodes: child sections (nested)

In [7]:
# fetch tree
tree_result = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get('result', [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 1

🌲 Raw tree (first node):
{
  "title": "Healthcare Data Ingestion and Migration",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "# Healthcare Data Ingestion and Migration\n\nA PROJECT REPORT\n\nSubmitted in partial fulfilment\nof the\nrequirement for the award of the degree of\n\nBACHELOR OF TECHNOLOGY (B. Tech)\n\nin\n\nComputer Science and Engineering (AIML)\n\nby\n\nYash Prasad\n\n229309105\n\nUnder the supervision of\n\nMr.Ajay Kumar Tyagi\nVeersa Technologies\n\n![img-0.jpeg](img-0.jpeg)\n\nMANIPAL UNIVERSITY\nJAIPUR\n\n(University under Section 2(f) of the UGC Act)\n\nDEPARTMENT OF ARTIFICIAL INTELLIGENCE\nAND MACHINE LEARNING,\nMANIPAL UNIVERSITY JAIPUR,\nRAJASTHAN, INDIA-303007\n\nJuly, 2026\n\nDate: 10$^{th}$ March 2026\n",
  "text": "# Healthcare Data Ingestion and Migration\n\nA PROJECT REPORT\n\nSubmitted in partial fulfilment\nof the\nrequirement for the award of the degree of\n\nBACHELOR OF TECHNOLOGY (B. Tech)\n\nin\n\nComputer Scien

In [8]:
# pretty-printing the full tree
def print_tree(nodes, indent=0):
    '''Recursively prints tree title'''
    for node in nodes:
        prefix = " "*indent + ('--' if indent > 0 else "")
        page = node.get('page_index', '?')
        print(f"{prefix}[{node['node_id']}] {node['title']} (p.{page})")
        if node.get('nodes'):
            print_tree(node['nodes'], indent + 1)

print('Full document structure: ')
print_tree(pageindex_tree)

Full document structure: 
[0000] Healthcare Data Ingestion and Migration (p.1)
 --[0001] CERTIFICATE (p.2)
  --[0002] ACKNOWLEDGMENTS (p.4)
  --[0003] LIST OF TABLES (p.6)
  --[0004] LIST OF FIGURES (p.7)
  --[0005] Contents (p.8)
  --[0006] 1 Introduction (p.9)
   --[0007] 1.1 Scope of the Work (p.10)
   --[0008] 1.2 Product Scenario (p.11)
  --[0009] 2 Requirement Analysis (p.13)
  --[0010] 1. Source Data Access and Extraction (p.13)
  --[0011] 2. Data Ingestion from Source Systems (p.13)
  --[0012] 3. Support for Multiple Ingestion Modes (p.13)
  --[0013] 4. Scheduling of Ingestion Workflows (p.13)
  --[0014] 5. Pipeline Extraction from SQL Server (p.13)
  --[0015] 6. Pipeline Configuration Generation (p.14)
  --[0016] 7. Databricks Asset Bundle Creation (p.14)
  --[0017] 8. Multi-Environment Deployment (p.14)
  --[0018] 9. Source Mart Generation (p.14)
  --[0019] 10. Data Mart Generation (p.14)
  --[0020] 11. Data Validation and Reconciliation (p.14)
  --[0021] 12. Monitoring and W

In [9]:
# count total nodes
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get('nodes'):
            total += count_nodes(n['nodes'])
    return total

total = count_nodes(pageindex_tree)
print('Total nodes in tree: ',total)

Total nodes in tree:  46


# LLM Tree Search

pageindex_retrieval:
    query + tree -> LLM reasons -> "node 34 and 32 contain the answer"

In [24]:
# from langchain_groq import ChatGroq
from groq import Groq
from dotenv import load_dotenv
load_dotenv()

os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

# llm Tree search function
def llm_tree_search(query: str, tree: list, model:str = 'llama-3.1-8b-instant') -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids

    Returns: dict with 'thinking' and 'node_list'
    """

    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                'node_id': n['node_id'],
                'title': n['title'],
                'page': n.get('page_index', '?'),
                'summary': n.get('text', '')[:200]
            }
            if n.get('nodes'):
                entry['children'] = compress(n['nodes'])
            out.append(entry)
        return out

    compressed_tree = compress(tree)

    prompt=  f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""
    
    # client = ChatGroq(
    #     model='llama-3.1-8b-instant',
    #     temperature=0,
    # )
    # # model_structured_output = client.with_structured_output(Output)
    # response = client.invoke(prompt)

    client = Groq()
    response = client.chat.completions.create(
        model=model,
        messages = [{'role':'user', 'content': prompt}],
        response_format={'type': 'json_object'}
    )
    result =json.loads(response.choices[0].message.content)
    return result

In [27]:
# sample query test
query = 'What is databricks?'
results = llm_tree_search(query, pageindex_tree)
print(results)


print("🧠 LLM Reasoning:")
print(results.get('thinking', 'NA'))
print()
print("🎯 Selected Node IDs:", results.get('node_list', []))

{'thinking': "The query asks about Databricks. I will start by checking the Table of Contents to identify the most relevant sections related to Databricks. Since the term 'Databricks' is likely to be mentioned in the 'Methodology' or 'Implementation' sections of the report, I will search for these sections. Within these sections, I will check for specific keywords related to Databricks such as 'Spark' or 'Databricks Cloud Platform'. I will then identify the node IDs corresponding to these sections to determine which contain the answer to the query.", 'node_list': ['0011', '0016']}
🧠 LLM Reasoning:
The query asks about Databricks. I will start by checking the Table of Contents to identify the most relevant sections related to Databricks. Since the term 'Databricks' is likely to be mentioned in the 'Methodology' or 'Implementation' sections of the report, I will search for these sections. Within these sections, I will check for specific keywords related to Databricks such as 'Spark' or '

In [28]:
# sample query test
query = 'What are penguins?'
results = llm_tree_search(query, pageindex_tree)
# print(results)


print("🧠 LLM Reasoning:")
print(results.get('thinking', 'NA'))
print()
print("🎯 Selected Node IDs:", results.get('node_list', []))

🧠 LLM Reasoning:
The query 'What are penguins?' contains keywords unrelated to the provided document tree. However, to maintain objectivity, I will analyze each node for any resemblance to the term 'penguin'. Given the document's focus on healthcare data ingestion and migration, a likely relevant section could be under 'System Design' where 'Data Integrity' or 'Data Validation and Reconciliation' might provide clues or tangential information about a penguin. Nonetheless, given the context of the information presented, the task seems to be looking for any hint of data ingestion or processing and how these might pertain to the query. In the given tree, there is '1 Introduction' which contains '2.2 Non-Functional Requirements' where 'Scalability' is discussed. Perhaps the idea of penguins as efficient swimmers could be drawn upon in this context.

🎯 Selected Node IDs: ['0037', '0027', '0028', '0035', '0041']


# End to End RAG Pipeline

In [29]:
# Helper: Finds nodes by ID
def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    '''Recursively walk tree and collect nodes matchinging the target_ids'''
    found = []
    for node in tree:
        if node['node_id'] in target_ids:
            found.append(node)
        if node.get('nodes'):
            found.extend(find_nodes_by_ids(node['nodes'], target_ids))
    return found

In [30]:
# generate answer from retrieved nodes
def generate_answer(query: str, nodes: list, model: str ='llama-3.1-8b-instant') -> str:
    """
    Input: Retrieved nodes 
    Output: answer with title and page numbers as citations 
    """

    if not nodes:
        return "No relevant sections found in the document"

    # build content string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = '\n\n---\n\n'.join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    client = Groq()
    response = client.chat.completions.create(
        model=model,
        messages = [{'role':'user', 'content': prompt}]
    )
    return response.choices[0].message.content

In [31]:
def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    if verbose:
        print(f'{'='*55}')
        print(f'Query: {query}')
        print(f'{'='*55}')
    
    # step 1 Tree search 
    search_result = llm_tree_search(query, tree)
    node_ids = search_result.get('node_list', [])

    if verbose:
        print(f'\n Reasoning: {search_result.get('thinking', '')[:200]}...')
        print(f'Retrieved node IDs: {node_ids}')

    # step 2: Retrieved Nodes
    nodes = find_nodes_by_ids(tree, node_ids)

    if verbose:
        print(f"Sections found: {[n['title'] for n in nodes]}")
    
    # generate answer 
    answer = generate_answer(query, nodes)

    if verbose:
        print(f'\n Answer: \n {answer}')
    
    return answer

In [32]:
answer = vectorless_rag(
    query='what are the phases of data ingestion?',
    tree = pageindex_tree
)

Query: what are the phases of data ingestion?

 Reasoning: To identify the phases of data ingestion, the search should start with the main chapters related to data ingestion and migration. The 'Data Ingestion and Migration' chapter may contain a general overv...
Retrieved node IDs: ['0010', '0011', '0013']
Sections found: ['1. Source Data Access and Extraction', '2. Data Ingestion from Source Systems', '4. Scheduling of Ingestion Workflows']

 Answer: 
 Based on the provided context, the phases of data ingestion are not explicitly stated. However, we can infer the following phases are involved in the data ingestion process:

1. Source Data Access and Extraction (Section '1. Source Data Access and Extraction' | Page 13) 
   * This phase involves secure access to operational datasets and inspection using SQL Server Management Studio (SSMS).

2. Data Ingestion from Source Systems (Section '2. Data Ingestion from Source Systems' | Page 13)
   * This phase involves automated extraction and 

# Expert Guided Retrieval

In [33]:
DATA_EXPERT_RULES = """
Route queries to the correct modeule using these rules:


Cross -cutting rules:

"""

In [34]:
from groq import Groq

def llm_tree_search_with_expert(
    query: str,
    tree: list,
    expert_rules: str,
    model: str = 'llama-3.1-8b-instant'
) -> dict:
    
    '''
    LLM tree search with domain expert rules injected
    '''

    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                'node_id': n['node_id'], 'title': n['title'],
                'page': n.get('page_index', '?'),
                'summary': n.get('text', '')[:150]
                }
            if n.get('nodes'):
                entry['children'] = compress(n['nodes'])
            out.append(entry)
        return out
    
    prompt = f"""You are a domain expert analyzing a document.
Find all node IDs that most likely contain the answer to the query.
Use the expert routing rules below to guide your reasoning.

Query: {query}

Document Tree:
{json.dumps(compress(tree), indent=2)}

Expert Routing Rules (follow these carefully):
{expert_rules}

Reply ONLY in this JSON format:
{{
  "thinking": "<your reasoning, referencing the expert rules>",
  "node_list": ["node_id1", "node_id2"]
}}"""
    groq_client = Groq()
    response = groq_client.chat.completion(
        model = model,
        messages= [{'role': 'user', 'content': prompt}],
        response_format = {'type': 'json_object'}
    )

    return json.loads(response.choices[0].message.content)

In [ ]:

query = "Details of the data ingestion?"

print(f"🔍 Query: {query}\n")

# Without expert rules
print("── Without Expert Rules ──")
basic   = llm_tree_search(query, pageindex_tree)
print("Nodes:", basic.get("node_list"))

print()

# With expert rules
print("── With Expert Rules ──")
guided  = llm_tree_search_with_expert(query, pageindex_tree, FINANCIAL_EXPERT_RULES)
print("Nodes:", guided.get("node_list"))
print("Reasoning:", guided.get("thinking", "")[:300])

In [35]:
# full expert guided RAG

def expert_rag(query: str, tree: list, rules: str) -> str:
    result = llm_tree_search_with_expert(query, tree, rules)
    nodes = find_nodes_by_ids(tree, result.get('node_list', []))
    return generate_answer(query, nodes)


# Chat API - Zero LLM Setup


In [37]:
# using PageIndex API (runs llm internally)

query = 'what are the steps of data ingestion?'
response = pi_client.chat_completions(
    messages = [{'role': 'user', 'content': query}],
    doc_id = doc_id
)

answer = response['choices'][0]['message']['content']
print('PageIndex Chat API Answer: ')
print(answer)

PageIndex Chat API Answer: 
Here are the **steps of data ingestion** as outlined in the report, organized across **7 phases**:

---

## Phase 1: Source Configuration & Ingestion Setup

**Step 1 – Source Creation and Configuration**
Operational datasets in SQL Server are identified and configured as sources. Schema info, metadata, and ingestion parameters are defined.

**Step 2 – Entity Registration in Ignite**
Source datasets (entities) are registered within the Ignite platform for ingestion tracking.

**Step 3 – Ingestion Mode Selection**
Each dataset is assigned an ingestion mode based on its update pattern:
- *Append, Incremental, Backfill, Full Load, or Replace*

---

## Phase 2: Data Ingestion

**Step 4 – Scheduling of Ingestion Jobs**
Ingestion workflows are scheduled via the Ignite Operations Console (e.g., DOS tables monthly, vendor tables daily/weekly).

**Step 5 – Raw Data Extraction**
At the scheduled time, workflows extract raw datasets from SQL Server.

**Step 6 – Data Tra

In [39]:
# Cleanup

# delete document from cloud
pi_client.delete_document(doc_id)

{'message': 'Document deleted successfully.'}